# ML Research Assistant — Capstone Project

## Domain
Machine Learning Research Assistant for answering questions from research papers and ML concepts.

## Problem Statement
Students and researchers often struggle to extract key insights from lengthy ML research papers. This system enables users to upload PDFs and ask questions, receiving grounded, context-aware answers.

## Success Criteria
- Accurate answers grounded in provided documents
- Ability to handle multi-turn conversations
- Robustness against hallucination and prompt injection
- Evaluation using RAGAS-style metrics

## Knowledge Base

The system uses a hybrid knowledge base:

1. Preloaded ML Topics
   - XGBoost, Random Forest, SVM, KNN
   - SHAP, LIME explainability
   - Evaluation metrics
   - Feature engineering
   - Decision tree
   - Neural networks
   - CNN
   - RNN
   - Transformers
   - Data preprocessing
   - Cross validation
   - Overfitting
   - Underfitting
   - Bias-variance tradeoff
   - Regularisation
   - Hyperparmeter Tuning
   - Ensemble learning
   - Clustering
   - Anomaly Detection
   - Model Comparision

2. User-uploaded PDFs
   - Research papers
   - Extracted using PyPDF
   - Limited to first 2000 characters for efficiency

3. Vector Database
   - ChromaDB (EphemeralClient)
   - SentenceTransformers embeddings
   - Semantic retrieval of relevant chunks

Agent was successfully built and tested in the local development environment.

All test cases passed (10/10), and evaluation metrics were computed separately.

## Architecture (LangGraph)

The system is built using LangGraph with the following nodes:

1. Memory Node
   - Maintains conversation history
   - Uses thread_id for session tracking

2. Router Node
   - Routes all queries to retrieval pipeline

3. Retrieval Node
   - Converts query → embedding
   - Retrieves top-k relevant documents from ChromaDB

4. Answer Node
   - Generates response using LLM (Groq)
   - Strict grounding rules:
     - Only use retrieved context
     - Reject out-of-scope queries
     - Prevent hallucination

5. Evaluation Node
   - Computes faithfulness score

6. Save Node
   - Stores conversation in memory

### Key Design Choices
- Sliding window memory (last 6 messages)
- Context-first answering
- Guardrails against prompt injection

## Test Results

The system was evaluated using 10 test cases:

- 8 domain-specific questions
- 2 red-team (adversarial) tests

### Results

RESULTS: 10/10 passed

### Observations
- All domain queries retrieved correct sources
- Out-of-scope queries correctly returned:
  "I don't know based on the provided documents."
- Prompt injection attempts were successfully rejected

## RAGAS Baseline Evaluation

Method: Manual LLM-based scoring (fallback method using Groq)

### Results

- Faithfulness: 0.94
- Answer Relevancy: 0.96
- Context Precision: 0.68

### Interpretation

- Faithfulness (0.94): High grounding in retrieved context, minimal hallucination
- Answer Relevancy (0.96): Answers directly address user queries
- Context Precision (0.68): Some irrelevant context retrieved due to broad semantic matching

### Analysis

The slightly lower context precision is due to retrieving multiple semantically similar documents. This can be improved by:
- Reducing top-k retrieval
- Adding reranking
- Applying stricter similarity thresholds

## Streamlit Application

The system includes a user-friendly Streamlit interface:

### Features
- PDF upload (multiple files)
- Chat-based interaction
- Source attribution display
- Faithfulness score visualization
- Conversation memory
- "New Conversation" reset button

### Workflow
1. Upload PDF(s)
2. Ask question
3. System retrieves relevant chunks
4. LLM generates grounded answer

## Engineering Decisions

1. ChromaDB EphemeralClient
   - Fast in-memory retrieval
   - No persistence overhead

2. SentenceTransformers Embeddings
   - Lightweight and efficient
   - Good semantic similarity

3. Strict Prompt Design
   - Prevents hallucination
   - Enforces grounded responses

4. Out-of-Scope Handling
   - Returns fallback instead of guessing
   - Improves reliability

5. Sliding Window Memory
   - Maintains context without overload
   - Efficient conversation tracking

6. Manual RAGAS Evaluation
   - Avoids dependency on paid APIs
   - Uses Groq for scoring

## Future Improvement

With more time, the system can be enhanced by:

- Adding a reranking model to improve retrieval precision
- Integrating web search tools for real-time knowledge
- Implementing multi-document reasoning
- Supporting longer document chunking strategies

## Files Included

- agent.py → LangGraph agent logic
- app.py → Streamlit interface
- test.py → Test suite (10 cases)
- eval.py → Manual RAGAS evaluation
- ragas_baseline.json → Evaluation results

## Run Instructions

```bash
pip install -r requirements.txt
set GROQ_API_KEY=your_key
streamlit run app.py